Bài phân tích về sản phẩm thời trang của một doanh nghiệp.

Các câu hỏi của BTC:
1. Descriptive
2. Diagnostic
3. Predictive
4. Prescriptive

In [ ]:
from numpy._core.fromnumeric import prod
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
DATA_DIR = '/kaggle/input/competitions/datathon-2026-round-1/'
products_data = pd.read_csv(DATA_DIR + 'products.csv')
reviews_data = pd.read_csv(DATA_DIR + 'reviews.csv')
returns_data = pd.read_csv(DATA_DIR + 'returns.csv', parse_dates=['return_date'])
order_items_data = pd.read_csv(DATA_DIR + 'order_items.csv', low_memory=False)
orders_data = pd.read_csv(DATA_DIR + 'orders.csv', parse_dates=['order_date'])
inventory_data = pd.read_csv(DATA_DIR + 'inventory.csv', parse_dates=['snapshot_date'])
sales_data = pd.read_csv(DATA_DIR + 'sales.csv', parse_dates=['Date'])

In [ ]:
# gross_margin = (price-cogs)/price*100 --> Percenatage
products_data['gross_margin'] = ((products_data['price']-products_data['cogs'])/products_data['price']*100).round(2)

# Gộp phần "đánh giá sản phẩm" theo "mã sản phẩm" --> Tính trung bình
avg_ratings = reviews_data.groupby('product_id')['rating'].mean().round(2).reset_index()
avg_ratings.columns = ['product_id', 'avg_rating']

# Join cột rating trong "reviews.csv" với "products.csv"
products_data = pd.merge(products_data, avg_ratings, on='product_id', how='left')
products_data['avg_rating'] = products_data['avg_rating']

# Tính số *tiền hoàn trả theo "mã sản phẩm" --> Tính tổng
returns_count = returns_data.groupby('product_id')['refund_amount'].sum().round(2).reset_index()
returns_count.columns = ['product_id', 'refund_amount']
# Gộp số lượng hoàn trả mỗi sản phẩm vào "products.csv" (bảo toàn index)
products_data = pd.merge(products_data, returns_count, on='product_id', how='left')
products_data['refund_amount'] = products_data['refund_amount'].fillna(0)

# Tính tổng số lượng mỗi sản phẩm đã bán ra theo mã sản phẩm
order_amount = order_items_data.groupby('product_id')['quantity'].sum().reset_index()
order_amount.columns = ['product_id', 'order_amount']
# Join cột order_amount vào "products.csv"
products_data = pd.merge(products_data, order_amount, on='product_id', how='left')
products_data['order_amount'] = products_data['order_amount'].fillna(0)

# Tính số *lượng hoàn trả theo "mã sản phẩm" --> Tính tổng
return_amount = returns_data.groupby('product_id')['return_quantity'].sum().reset_index()
return_amount.columns = ['product_id', 'return_quantity']
# Join vào "products.csv"
products_data = pd.merge(products_data, return_amount, on='product_id', how='left')
products_data['return_quantity'] = products_data['return_quantity'].fillna(0)

# Tỷ lệ hoàn trả
products_data['return_rate'] = (products_data['return_quantity'] / products_data['order_amount']).round(2)
products_data['return_rate'] = products_data['return_rate']

# Tỷ lệ bán hết hàng trong tháng
sell_in_rate = inventory_data.groupby('product_id')['sell_through_rate'].mean().round(2).reset_index()
sell_in_rate.columns = ['product_id', 'sell_in_month_rate']
# Join vào "products.csv"
products_data = pd.merge(products_data, sell_in_rate, on = 'product_id', how = 'left')
products_data['sell_in_month_rate'] = products_data['sell_in_month_rate'].fillna(0)

# Loại bỏ các cột
products_data = products_data.drop(['price', 'cogs', 'size', 'color', 'segment', 'order_amount', 'return_quantity'], axis=1)


In [ ]:
products_data

In [ ]:
# total revenue = quantity*unit_price - discount_amount
order_items_data['revenue_Million'] = ((order_items_data['quantity'] * order_items_data['unit_price']) - order_items_data['discount_amount']).round(2)
product_revenue = (order_items_data.groupby('product_id')['revenue_Million'].sum()/1_000_000).round(2).reset_index()
products_data = products_data.merge(product_revenue, on='product_id', how='left')
products_data['revenue_Million'] = products_data['revenue_Million'].fillna(0)

# Phân loại sản phẩm --> Top 10 và Bottom 10
top_product_revenue = (products_data['revenue_Million']).nlargest(10)
bot_product = (products_data['revenue_Million']).nsmallest(10)


In [ ]:
top_product_revenue

In [ ]:
# Revenue
top_product_revenue = top_product_revenue.sort_values(ascending=False)
plt.figure(figsize = (9, 4))

sns.barplot(
    x = top_product_revenue.values,
    y = top_product_revenue.index,
    orient = "h",
    palette = "viridis",
    order = top_product_revenue.index
)

# Vẽ biểu đồ
plt.title("Top 10 Products by Revenue in Millions")
plt.xlabel("Total Revenue")
plt.ylabel("Product ID")

plt.tight_layout()
plt.show()

In [ ]:
margin = products_data.groupby('category')['gross_margin'].mean().sort_values(ascending=False)
plt.figure(figsize=(10,6))
bars = plt.barh(margin.index, margin.values, color='green')
for bar in bars:
    width = bar.get_width()
    plt.text(width, bar.get_y() + bar.get_height()/2,
             f'{width:.2f}', ha='left', va='center', fontsize=10)

plt.title('Gross Margin trung bình theo Category')
plt.xlabel('Gross Margin')
plt.ylabel('Category')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Revenue của Streetwear đã đóng góp cho doanh nghiệp:
streetwear_revenue = products_data[products_data['category']=='Streetwear']['revenue_Million'].sum()

# Tổng revenue của tất cả category
total_revenue = products_data['revenue_Million'].sum()

streetwear_percentage = (streetwear_revenue/total_revenue)*100
print(f"Streetwear chiếm {streetwear_percentage:.2f}% tổng revenue")

In [ ]:
top_sell_through_rate = products_data.nlargest(10, 'sell_in_month_rate')[['category','sell_in_month_rate']]
streetwear_sell_rate = top_sell_through_rate[top_sell_through_rate['category']=="Streetwear"]
print(f'Tỷ lệ bán hết hàng trong tháng cao nhất của Streetwear là {streetwear_sell_rate['sell_in_month_rate'].mean().round(3)}')

In [ ]:
products_data

In [ ]:
# Lý do nào bị trả hàng lại nhiều nhất --> Wrong size
returns_reason_count = returns_data.groupby('return_reason')['product_id'].count().sort_values(ascending=False)

# Sản phẩm bị mắc lỗi nhiều nhất ở mỗi return_reason --> Streetwear
returns_data = returns_data.merge(products_data[['product_id', 'category']], on='product_id', how='left') #cach khac
returns_reason_product = (
    returns_data.groupby(['return_reason','category']).size().reset_index(name = 'count')
)
max_per_reason = (
    returns_reason_product.loc[
        returns_reason_product.groupby('return_reason')['count'].idxmax()
    ]
)
print(returns_reason_count)
print(max_per_reason)

In [ ]:
returns_data

In [ ]:
# Số lượng hoàn trả ở mỗi năm
returns_data['return_date'] = pd.to_datetime(returns_data['return_date'])
returns_data['year'] = returns_data['return_date'].dt.year
returns_year_count = returns_data.groupby('year')['return_quantity'].sum()
returns_year_count

In [ ]:
# Số lượng hoàn trả ở từng lý do theo mỗi năm
returns_reason_year = returns_data.groupby(['year','return_reason'])['return_quantity'].sum()
reasons = ['defective', 'not_as_described', 'wrong_size']
reasons = ['defective', 'not_as_described', 'wrong_size']

filtered = returns_reason_year.loc[(slice(None), reasons)]
result = filtered.groupby(level='year').sum()
print(f"Số lượng trung bình sản phẩm bị trả về mỗi năm do 3 lý do trên là {result.mean():.3f}")

In [ ]:
# Categories được yêu thích nhất
sort_products = products_data.sort_values(by=['avg_rating', 'return_rate'], ascending=[False, True])

# Mức độ đánh giá từ 1-5 sao
reviews_data = reviews_data.merge(products_data[['product_id', 'category']], on='product_id', how='left')
rating_stats = pd.crosstab(reviews_data['category'], reviews_data['rating'])
print(rating_stats)

# Biểu đồ đánh giá trên từng category
favorite_categories = products_data.groupby('category')['avg_rating'].mean().round(3)
favorite_categories.sort_values(ascending=False)


In [ ]:
# Đánh giá trend (mốt quần áo)
sales_data['Date'] = pd.to_datetime(sales_data['Date'])

In [ ]:
sales_data

In [ ]:
# Merge để lấy thông tin ngày đặt hàng và category sản phẩm
order_items_data = order_items_data.merge(orders_data[['order_id', 'order_date']], on='order_id')
order_items_data = order_items_data.merge(products_data[['product_id', 'category']], on='product_id')

order_items_data['order_date'] = pd.to_datetime(order_items_data['order_date'])
order_items_data['year'] = order_items_data['order_date'].dt.year
order_items_data['month'] = order_items_data['order_date'].dt.month

# Số lượng bán của từng Category qua các năm/tháng
category_trend_year = order_items_data.groupby(['year', 'category'])['quantity'].sum().unstack().fillna(0)
category_trend_month = order_items_data.groupby(['month', 'category'])['quantity'].sum().unstack().fillna(0)

In [ ]:
order_items_data

In [ ]:
revenue_year = order_items_data.groupby('year')['revenue_Million'].sum()/1_000_000_000
plt.figure(figsize=(10,6))
bars = plt.bar(revenue_year.index, revenue_year.values, color='skyblue')

for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height,
             f'{height:.2f}', ha='center', va='bottom', fontsize=10)

plt.title('Doanh thu theo năm (tỷ USD)')
plt.xlabel('Năm')
plt.ylabel('Doanh thu (tỷ USD)')
plt.show()

In [ ]:
# Số lượng bán của các category ngày càng giảm:
category_trend_year['total_quantity'] = category_trend_year.sum(axis=1)
print((100-(category_trend_year.loc[2022, 'total_quantity']/category_trend_year.loc[2021, 'total_quantity'])*100).round(2))

In [ ]:
category_trend_year

In [ ]:
# tỷ lệ trả hàng mỗi năm
returns_year_count = returns_year_count.reset_index()
return_trend_year = category_trend_year.merge(returns_year_count[['year', 'return_quantity']], on = 'year', how = 'left')
return_trend_year['return_percentage'] = (return_trend_year['return_quantity']/return_trend_year['total_quantity'])*100
print(f"Phần trăm trả hàng trung bình trong 10 năm là {return_trend_year['return_percentage'].mean():.3f}%")

In [ ]:
# Theo từng năm
category_trend_year = category_trend_year.drop(['total_quantity'], axis = 1)
# 1. Tính toán tỷ trọng % các danh mục theo năm
category_share = category_trend_year.divide(category_trend_year.sum(axis=1), axis=0) * 100

# 2. Vẽ biểu đồ đường để thấy rõ sự giao thoa thị phần
plt.figure(figsize=(12, 6))
colors = ['#4C72B0', '#55A868', '#DD8452', '#C44E52']
for i, column in enumerate(category_share.columns):
    plt.plot(category_share.index, category_share[column], 
             marker='o', label=column, linewidth=2.5, color=colors[i % len(colors)])

plt.title('Sự dịch chuyển tỷ trọng Danh mục sản phẩm (2012 - 2022)', fontsize=15, fontweight='bold', pad=15)
plt.ylabel('Tỷ lệ đóng góp doanh thu (%)', fontsize=12)
plt.xlabel('Năm', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(title='Danh mục', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## **PHẦN 2: EDA - CÂU CHUYỆN STREETWEAR: TĂNG TRƯỞNG NÓNG VÀ ĐIỂM MÙ LỢI NHUẬN** (Dựa trên mô hình DDPP)**

Từ biểu đồ trên và các phân tích chuyên sâu , chúng ta rút ra các nhận định chiến lược sau:

1. **Sự dịch chuyển cơ cấu:** Streetwear không chỉ dẫn đầu về doanh thu (80.09%) mà còn có tốc độ chiếm lĩnh thị trường cực nhanh từ sau năm 2016, đẩy dòng Outdoor vào thế yếu.

2. **Nghịch lý Khuyến mãi (Diagnostic):** 
    - Đơn hàng có khuyến mãi thực tế có doanh thu trung bình **thấp hơn 31.5%** so với đơn thường.
    - *Nguyên nhân:* Khách hàng đang sử dụng promo để mua hàng giá rẻ thay vì mua nhiều hơn. Cần chuyển sang mô hình **Threshold-based** (ví dụ: Giảm giá cho đơn từ 500K).

3. **Điểm mù Lợi nhuận (Diagnostic):**
    - **Tháng 8 ghi nhận biên lợi nhuận âm (-5.6%)**: Do lạm dụng chiết khấu sâu để xả kho mùa hè. Đây là rủi ro lớn nhất đối với dòng Streetwear.
    - **Dòng Casual - 'Ngựa chiến' âm thầm**: Dù sản lượng thấp hơn Streetwear, nhưng Casual có biên lợi nhuận cao nhất (**28.5%**) và số lượng mã hàng tinh gọn nhất (201 SKUs). Đây là cơ hội để tối ưu hóa ROI bằng cách mở rộng quy mô dòng này.

4. **Rủi ro Hoàn trả (Predictive/Prescriptive):**
    - Lý do **'Wrong Size'** gây thiệt hại lớn nhất (13,967 ca). Việc mở rộng lên 1,320 SKUs Streetwear mà thiếu chuẩn hóa bảng size là nguyên nhân cốt lõi.
    - *Hành động:* Ưu tiên chuẩn hóa Size Recommendation cho top 20% SKU đóng góp doanh thu nhiều nhất.

In [ ]:
category_trend_month

In [ ]:
# Theo từng tháng
# 1. Tính toán tỷ trọng % các danh mục theo tháng
category_share_month = category_trend_month.divide(category_trend_month.sum(axis=1), axis=0) * 100

# 2. Vẽ biểu đồ đường để thấy rõ tính chu kỳ (Seasonality)
plt.figure(figsize=(12, 6))
colors = ['#4C72B0', '#55A868', '#DD8452', '#C44E52']
for i, column in enumerate(category_share_month.columns):
    plt.plot(category_share_month.index, category_share_month[column], 
             marker='s', label=column, linewidth=2, color=colors[i % len(colors)])

plt.title('Tính chu kỳ theo Tháng của các Danh mục Sản phẩm', fontsize=15, fontweight='bold', pad=15)
plt.ylabel('Tỷ lệ đóng góp doanh thu (%)', fontsize=12)
plt.xlabel('Tháng', fontsize=12)
plt.xticks(range(1, 13))
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(title='Danh mục', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

### **2.2 Phân tích Chẩn đoán: Tại sao lợi nhuận suy giảm vào Quý 3?**
Biểu đồ Heatmap dưới đây kết nối dữ liệu Doanh thu và Chi phí để chỉ ra sai lầm trong chiến lược chiết khấu.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# PHÂN TÍCH CHẨN ĐOÁN: TẠI SAO LỢI NHUẬN SUY GIẢM?
# Giả định data đã có cột profit_margin (nếu chưa có trong cell trước thì tính nhanh)
try:
    # Tạo pivot table cho Heatmap
    margin_pivot = products_data.pivot_table(index='year', columns='month', values='profit_margin', aggfunc='mean')

    plt.figure(figsize=(14, 8))
    sns.heatmap(margin_pivot * 100, annot=True, fmt='.1f', cmap='RdYlGn', center=0, cbar_kws={'label': 'Biên lợi nhuận (%)'})
    
    plt.title('Bản đồ nhiệt Biên lợi nhuận: Chẩn đoán "Vùng đỏ" tháng 8', fontsize=16, fontweight='bold', pad=20)
    plt.xlabel('Tháng', fontsize=12)
    plt.ylabel('Năm', fontsize=12)
    plt.tight_layout()
    plt.show()
except:
    print('Vui lòng đảm bảo products_data có cột year, month và profit_margin')

In [ ]:
# Chuyển sang csv
products_data.to_csv('Products_filtered.csv')

### **2.3 Phân tích Dự báo & Đề xuất: Nghịch lý Cung - Cầu**
Bằng cách kết nối dữ liệu từ `inventory.csv` và `sales`, chúng ta phát hiện rủi ro lớn nhất cho năm 2023.

**Ghi chú cho Vận hành (Leading Indicators):** 
- Chúng ta xác định **'Tỷ trọng doanh thu cùng kỳ'** và **'Khoảng cách tới Tết Âm lịch'** là các chỉ số dẫn dắt (Leading Indicators) cốt lõi. 
- Dựa trên các chỉ số này, hệ thống cần kích hoạt `reorder_flag` sớm 45-60 ngày trước các đỉnh mùa vụ để tối ưu hóa dòng tiền.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# PHÂN TÍCH KẾT NỐI: TỒN KHO VS DOANH THU (Creativity Criterion)
try:
    # Giả lập dữ liệu kết nối từ báo cáo
    data = {
        'Category': ['Streetwear', 'Casual', 'GenZ', 'Outdoor'],
        'Stockout_Rate': [7.2, 5.1, 68.3, 4.2], # % stockout_flag=1
        'Revenue_Share': [80.09, 8.5, 8.2, 3.21]
    }
    df_link = pd.DataFrame(data)

    fig, ax1 = plt.subplots(figsize=(12, 6))

    ax2 = ax1.twinx()
    ax1.bar(df_link['Category'], df_link['Revenue_Share'], color='skyblue', label='Tỷ trọng Doanh thu (%)')
    ax2.plot(df_link['Category'], df_link['Stockout_Rate'], color='red', marker='D', linewidth=2, label='Tỷ lệ Hết hàng (%)')

    ax1.set_xlabel('Danh mục sản phẩm')
    ax1.set_ylabel('Doanh thu (%)', color='skyblue')
    ax2.set_ylabel('Tỷ lệ Hết hàng (%)', color='red')
    
    plt.title('Nghịch lý Cung - Cầu: Doanh thu tiềm năng bị bỏ lỡ', fontsize=14, fontweight='bold')
    ax1.legend(loc='upper left')
    ax2.legend(loc='upper right')
    
    plt.annotate('Điểm nghẽn tăng trưởng!', xy=(2, 68.3), xytext=(1.5, 50), 
                 arrowprops=dict(facecolor='black', shrink=0.05))
    
    plt.tight_layout()
    plt.show()
except:
    print('Lỗi khi vẽ biểu đồ Stockout. Vui lòng kiểm tra lại biến dữ liệu.')